In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

from app.grid_utils import generate_grid, load_boundary


## 1. Load district boundary

Source: GADM v4.1, India, admin level 2 (`ADM_ADM_2`), downloaded to `data/raw/boundaries/gadm41_IND.gpkg` and filtered to the Kamrup Metropolitan feature, saved as `data/raw/boundaries/kamrup_metropolitan.geojson`.

In [ ]:
boundary = load_boundary("../data/raw/boundaries/kamrup_metropolitan.geojson")

print("CRS:", boundary.crs)
print("Rows:", len(boundary))
print(boundary[["NAME_1", "NAME_2", "ENGTYPE_2"]])
print("Bounds (minx, miny, maxx, maxy):", boundary.total_bounds)

## 2. Generate the 1km x 1km grid

`generate_grid` reprojects to EPSG:32646 (UTM 46N) to build true 1km cells, clips each cell to the district polygon, then returns geometry in EPSG:4326 for storage.

In [ ]:
grid = generate_grid(boundary, cell_size_m=1000, metric_crs="EPSG:32646")

print("Grid CRS:", grid.crs)
print("Number of grid cells:", len(grid))
print("Duplicate grid_id count:", grid["grid_id"].duplicated().sum())
grid.head()

In [ ]:
boundary_metric = boundary.to_crs("EPSG:32646")
district_area_km2 = boundary_metric.geometry.area.sum() / 1e6

print(f"District area (from boundary polygon): {district_area_km2:.1f} km^2")
print(f"Grid cell count: {len(grid)}")
print(f"Approx area implied by cell count: {len(grid) * 1.0:.1f} km^2 (at 1 km^2/cell)")

## 3. Plot grid over district boundary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

boundary.boundary.plot(ax=ax, color="black", linewidth=1.5, zorder=2)
grid.plot(ax=ax, facecolor="none", edgecolor="steelblue", linewidth=0.3, zorder=1)

ax.set_title(f"Kamrup Metropolitan - 1km Grid ({len(grid)} cells)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig("../docs/img/02_grid_preview.png", dpi=150)
plt.show()

## 4. Save grid to parquet

In [ ]:
output_path = "../data/processed/kamrup_metro_grid_1km.parquet"
grid.to_parquet(output_path)

print("Saved:", output_path)
print("Rows:", len(grid))
print("Columns:", list(grid.columns))